In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Column
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime as datum_vreme

## OpenAQ Sensors

In [0]:
bronze_openaq_measurements = spark.readStream.table(
    "bg_traffic.bg_traffic_bronze.openaq_measurements"
)

In [0]:
bronze_openaq_locations = spark.table("bg_traffic.bg_traffic_bronze.openaq_locations").orderBy(F.col("fetched_at").desc()).limit(1)

bronze_openaq_locations.display()

In [0]:
locations_flatten = bronze_openaq_locations\
    .select(
        F.explode("data.results").alias("location"),
        F.col("location.id").alias("location_id")
    
    )

locations_flatten = locations_flatten.select(
    F.explode("location.sensors").alias("sensor"),
    
).select(
        
        F.col("sensor.id").alias("sensor_id"),
        F.col("sensor.parameter.displayName").alias("sensor_name"),
        F.col("sensor.name").alias("sensor_code"),
        F.col("sensor.parameter.units").alias("units")
)

locations_flatten.display()

In [0]:
measurements_checkpoint = (
    "/Volumes/bg_traffic/bg_traffic_silver/checkpoints/openaq_measurements"
)

### Flattening

In [0]:
measurements_flatten = bronze_openaq_measurements.select(
    F.col("fetched_at"),
    F.col("_ingestion_timestamp"),
    F.col("_source_file"),
    F.explode("data.results").alias("measurement"),
).select(
    "fetched_at",
    "_ingestion_timestamp",
    "_source_file",
    F.col("measurement.locationsId").alias("location_id"),
    F.col("measurement.sensorsId").alias("sensor_id"),
    F.col("measurement.datetime.utc").alias("measurement_timestamp"),
    F.col("measurement.value").alias("measurement_value"),
)

### Casting

In [0]:
measurements_types = (
    measurements_flatten.withColumn("fetched_at", F.to_timestamp("fetched_at"))
    .withColumn("measurement_timestamp", F.to_timestamp("measurement_timestamp"))
    .withColumn("location_id", F.col("location_id").cast("long"))
    .withColumn("sensor_id", F.col("sensor_id").cast("long"))
    .withColumn("measurement_value", F.col("measurement_value").cast("double"))
)


measurements_types.printSchema()

### Validate

In [0]:
measurements_valid = measurements_types.filter(
    (F.col("location_id").isNotNull())
    & (F.col("sensor_id").isNotNull())
    & (F.col("measurement_timestamp").isNotNull())
    & (F.col("measurement_value").isNotNull())
    & (~F.isnan(F.col("measurement_value")))
)

In [0]:
measurements_valid = measurements_valid.alias("m").join(
    locations_flatten.alias("l"),
    F.col("m.sensor_id") == F.col("l.sensor_id"),
    "inner"
).select(
    F.col("m.location_id"),
    F.col("m.sensor_id"),
    F.col("m.measurement_timestamp"),
    F.col("m.measurement_value"),
    F.col("m.fetched_at"),
    F.coalesce(F.col("l.sensor_name"), F.lit("UNKNOWN")).alias("pollutant_name"),
    F.col("l.units").alias("measurement_units"),
    F.col("_ingestion_timestamp"),
    F.col("_source_file")
    
)

### Merge & Dedup

In [0]:
def merge_measurements(df_source, batch_id):
    if df_source.isEmpty():
        return

    measurements_window = Window.partitionBy(
        "location_id", "sensor_id", "measurement_timestamp"
    ).orderBy(F.col("fetched_at").desc())

    measurements_dedup = (
        df_source.withColumn("row_num", F.row_number().over(measurements_window))
        .filter(F.col("row_num") == 1)
        .drop("row_num")
    )

    silver_table_measurements = (
        DeltaTable.forName(spark, "bg_traffic.bg_traffic_silver.openaq_measurements")
        .alias("target")
        .merge(
            measurements_dedup.alias("source"),
            """
                            target.location_id = source.location_id
                            AND target.sensor_id = source.sensor_id
                            AND target.measurement_timestamp = source.measurement_timestamp
                            """,
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

In [0]:
query_measurements = (
    measurements_valid.writeStream.foreachBatch(merge_measurements)
    .option("checkpointLocation", measurements_checkpoint)
    .trigger(availableNow=True)
    .start()
)


query_measurements.awaitTermination()

In [0]:
%sql
SELECT
  *
FROM
  bg_traffic.bg_traffic_silver.openaq_locations;

In [0]:
%sql
SELECT
  *
FROM
  bg_traffic.bg_traffic_silver.openaq_measurements;